In [27]:
import numpy as np
import pandas as pd

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [29]:
df = pd.read_csv("/home/rgukt/Downloads/Titanic-Dataset.csv")
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [30]:
df = df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1)

In [31]:
df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [32]:
#Here we can see Sex and Embarked categorical (nominal data)
#Perform One hot encoding

In [33]:
#Checking null values
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [34]:
df["Sex"].value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

In [35]:
df.shape

(891, 8)

In [36]:
df["Embarked"].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [37]:
df["Survived"].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [38]:
#Training and testing datasets
x_train,x_test,y_train,y_test=train_test_split(df.drop(["Survived"],axis=1),df["Survived"],test_size=0.2,random_state=42)

In [43]:
x_train.shape

(712, 7)

In [44]:
x_train.head(3)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.500,S
733,2,male,23.0,0,0,13.000,S
382,3,male,32.0,0,0,7.925,S


In [51]:
#We have null values in 2 columns Age,Embarked
se_age=SimpleImputer()#by deafult nan replaced with mean
se_embarked = SimpleImputer(strategy = "most_frequent")#Here using most frequent values

x_train_age = se_age.fit_transform(x_train[["Age"]])
x_train_embarked = se_embarked.fit_transform(x_train[["Embarked"]])

x_test_age = se_age.transform(x_test[["Age"]])
x_test_embarked = se_embarked.transform(x_test[["Embarked"]])

In [52]:
#Applying one hot encoding on embarked and sex
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown="ignore")
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown="ignore")

x_train_sex = ohe_sex.fit_transform(x_train[["Sex"]])
x_train_embarked=ohe_embarked.fit_transform(x_train[["Embarked"]])

x_test_sex = ohe_sex.transform(x_test[["Sex"]])
x_test_embarked=ohe_embarked.transform(x_test[["Embarked"]])

In [53]:
x_train_rem = x_train.drop(["Age","Sex","Embarked"],axis=1)
x_test_rem = x_test.drop(["Age","Sex","Embarked"],axis=1)
x_train_trans = np.concatenate((x_train_age,x_train_sex,x_train_embarked,x_train_rem),axis=1)
x_test_trans = np.concatenate((x_test_age,x_test_sex,x_test_embarked,x_test_rem),axis=1)

In [54]:
from sklearn.tree import DecisionTreeClassifier
clf  = DecisionTreeClassifier()
clf.fit(x_train_trans,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [55]:
y_pred = clf.predict(x_test_trans)

In [56]:
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test,y_pred)
print(score)

0.7877094972067039


In [58]:
import pickle
pickle.dump(ohe_sex,open("titanic_dataset/models/ohe_sex.pkl","wb"))
pickle.dump(ohe_embarked,open("titanic_dataset/models/ohe_embarked.pkl","wb"))
pickle.dump(clf,open("titanic_dataset/models/clf.pkl","wb"))